# G3-VPR v3 (CVPR A*): Train + Ablation + Chấm SOTA + G-OTR + Conformal Cascade 100% Trên Kaggle (`T4 x2`)
**Toàn bộ nâng cấp chuẩn CVPR trong bản `v3`:**
- **Vá lỗi v2:** Bật `RandAugment + RandomGrayscale (20%) + ColorJitter` (khắc phục hoàn toàn lỗi rớt `AmsterTime`) + **cấm tuyệt đối rò rỉ tập Test** lúc chọn checkpoint.
- **3 Đột phá hợp nhất:** `G-OTR` (xác minh hình học 3D Top-20 không tốn FLOPs backbone), `Nyström-PMCI` (trừ nhiễu miền + SwAV chống chết nguyên mẫu), và `AM-VPR Conformal Cascade` (`CSD` + thoát sớm bảo chứng rủi ro).
- **Chấm điểm trực tiếp trên Kaggle:** Tự động chấm `G3-VPR v3` (`128..4096-D + G-OTR + Cascade`), so sánh **SOTA (`MegaLoc`, `DINOv2-SALAD`, `EigenPlaces`) ở `BS=1`**, chấm bảng **Ablation đồng ngân sách**, đo **FAISS 1M**, và xuất sẵn báo cáo **`CVPR_V3_SUMMARY_REPORT.md`** trong file ZIP Output!

**Cách chạy trên Kaggle:**
1. **Settings** bên phải: chọn **Accelerator → `GPU T4 x2`** và bật **Internet → `ON`**.
2. **Input** bên phải: bấm **`+ Add Input`** → gắn **`amaralibey/gsv-cities`** (nếu bạn có gắn thêm `pitts30k` / `sped` / `amstertime` / `msls` thì Cell 6 sẽ tự nhận và chấm luôn!).
3. Bấm **`Save Version` → `Save & Run All (Commit)`** rồi tải file `g3vpr_paper_artifacts_v3.zip` về đối chiếu!

In [ ]:
# Cell 1: Tu dong giai nen toan bo code G3-VPR v3 (CVPR All-in-One) ngay trong Kaggle/Colab
import base64, io, os, subprocess, sys, tarfile, torch

WORK_ROOT = "/kaggle/working/D2VPR" if os.path.isdir("/kaggle/working") else os.path.abspath("./D2VPR_cloud")
os.makedirs(WORK_ROOT, exist_ok=True)

ARCHIVE_B64 = (
    "..."
    "KBQKhULxX+MPjISKHQC4BgA="
)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ARCHIVE_B64)), mode="r:gz") as tf:
    tf.extractall(WORK_ROOT)
os.chdir(WORK_ROOT)
print("[OK] Da giai nen toan bo du an G3-VPR v3 (CVPR) tai:", os.getcwd())
print("PyTorch:", torch.__version__, "| CUDA GPU:", torch.cuda.is_available(), "| So GPU:", torch.cuda.device_count())
if not torch.cuda.is_available():
    print("CANH BAO: Ban chua bat GPU! Vao Settings ben phai -> Accelerator -> chon GPU T4 x2 va bat Internet ON!")
else:
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pytorch-metric-learning"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)
print("[OK] Moi truong thu vien G3-VPR v3 da san sang 100%!")

In [ ]:
# Cell 2: Tai trong so khoi dong am ViT-S+Adapter, DINOv2-S & Tu dong tim GSV-Cities + VPR test sets
import os, urllib.request
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

os.makedirs("./weights_init", exist_ok=True)
warm_ckpt = "./weights_init/vits_adapter_warmstart_fp16.pt"
if not os.path.isfile(warm_ckpt):
    for u in [
        "https://raw.githubusercontent.com/phuccode25/G3-VPR/main/weights_init/vits_adapter_warmstart_fp16.pt",
        "https://github.com/phuccode25/G3-VPR/raw/main/weights_init/vits_adapter_warmstart_fp16.pt",
    ]:
        try:
            print("Dang tai trong so khoi dong am ViT-S+Adapter tu:", u)
            urllib.request.urlretrieve(u, warm_ckpt)
            if os.path.isfile(warm_ckpt) and os.path.getsize(warm_ckpt) > 10_000_000:
                break
        except Exception as e:
            print("Thu URL tiep theo do:", e)
print("[OK] Warm-start ViT-S:", os.path.isfile(warm_ckpt))

dinov2_ckpt = "./dinov2_vits14_pretrain.pth"
if not os.path.isfile(dinov2_ckpt):
    print("Dang tai trong so DINOv2 ViT-S/14...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth",
        dinov2_ckpt
    )
print("[OK] DINOv2-S:", os.path.isfile(dinov2_ckpt))

GSV_PATH = None
if os.path.isdir("/kaggle/input"):
    for r, dirs, _ in os.walk("/kaggle/input"):
        if "Dataframes" in dirs and "Images" in dirs:
            GSV_PATH = r
            break
if GSV_PATH is None:
    import kagglehub
    GSV_PATH = kagglehub.dataset_download("amaralibey/gsv-cities")
print("[OK] Da tim thay GSV-Cities tai:", GSV_PATH)

In [ ]:
# Cell 3: Kiem tra kien truc G3-VPR v3 (Nystrom-PMCI + G-OTR), tinh bat bien BS=1 & toc do GPU T4
import os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")
subprocess.run([sys.executable, "-u", "experiments/exp2_architecture_verify_and_profile.py"], check=True)

In [ ]:
# Cell 4: Huan luyen Mo hinh Chinh G3-VPR v3 (M5_full) voi Data Aug + Nystrom-PMCI + CSD + Dual Teachers
import os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

SAVE_DIR = "./checkpoints/g3vpr_M5_full"
cmd = [
    sys.executable, "-u", "train_g3vpr.py",
    "--ablation", "M5_full",
    "--gsv_path", GSV_PATH,
    "--dinov2_path", "./dinov2_vits14_pretrain.pth",
    "--teacher_2d", "megaloc",
    "--teacher_3d", "depth_anything_v2",
    "--bs", "32",
    "--grad_accum", "2",
    "--epochs", "8",
    "--max_places", "22000",
    "--lr", "3e-4",
    "--unfreeze_last_n", "6",
    "--save_dir", SAVE_DIR,
]
if os.path.isfile("./weights_init/vits_adapter_warmstart_fp16.pt"):
    cmd.extend(["--warmstart_ckpt", "./weights_init/vits_adapter_warmstart_fp16.pt"])
if os.path.isfile(f"{SAVE_DIR}/g3vpr_last.pth"):
    cmd.extend(["--resume_ckpt", f"{SAVE_DIR}/g3vpr_last.pth"])

print("Running G3-VPR v3 M5_full:", " ".join(cmd))
subprocess.run(cmd, stdin=subprocess.DEVNULL, check=True)
if os.path.isfile(f"{SAVE_DIR}/g3vpr_last.pth"):
    os.remove(f"{SAVE_DIR}/g3vpr_last.pth")

In [ ]:
# Cell 5: Bang Ablation Dong Ngan Sach & Tach Sach Bien (CVPR Table 4: 4 ep x 6000 places cho moi dong)
import os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

ABLATION_SPECS = [
    ("M1", "M1", "none", "none", False),
    ("M2a_GatedDRM", "M2a", "none", "none", False),
    ("M2b_3DDGSA", "M2b", "none", "none", False),
    ("M3_NystromPMCI", "M3", "none", "none", False),
    ("M4a_2DKD_only", "M4a", "megaloc", "none", False),
    ("M4b_3DKD_only", "M4b", "none", "depth_anything_v2", False),
    ("M4c_DualKD", "M4c", "megaloc", "depth_anything_v2", False),
    ("M5_EqBudget", "M5_full", "megaloc", "depth_anything_v2", False),
    ("M5_NoWarmStart", "M5_full", "megaloc", "depth_anything_v2", True),
]
for tag, abl_cfg, t2d, t3d, no_ws in ABLATION_SPECS:
    out_d = f"./checkpoints/ablation_{tag}"
    abl_cmd = [
        sys.executable, "-u", "train_g3vpr.py",
        "--ablation", abl_cfg,
        "--gsv_path", GSV_PATH,
        "--cities", "all",
        "--dinov2_path", "./dinov2_vits14_pretrain.pth",
        "--teacher_2d", t2d,
        "--teacher_3d", t3d,
        "--bs", "32",
        "--grad_accum", "2",
        "--epochs", "4",
        "--max_places", "6000",
        "--lr", "3e-4",
        "--unfreeze_last_n", "6",
        "--save_dir", out_d,
    ]
    if no_ws:
        abl_cmd.append("--no_warmstart")
    elif os.path.isfile("./weights_init/vits_adapter_warmstart_fp16.pt"):
        abl_cmd.extend(["--warmstart_ckpt", "./weights_init/vits_adapter_warmstart_fp16.pt"])
    print(f"\n=== Running Equal-Budget Ablation {tag} ({abl_cfg}, 2D={t2d}, 3D={t3d}, no_ws={no_ws}) ===")
    subprocess.run(abl_cmd, stdin=subprocess.DEVNULL, check=True)
    for rm_fn in ("g3vpr_last.pth", "g3vpr_universal_best.pth"):
        p_rm = os.path.join(out_d, rm_fn)
        if os.path.isfile(p_rm):
            os.remove(p_rm)

In [ ]:
# Cell 6: CHAM DIEM TRUC TIEP TREN KAGGLE (G3-VPR v3 + G-OTR + Conformal Cascade + SOTA Hub + Ablations + FAISS 1M)
import os, shutil, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

m5_ckpt = "./checkpoints/g3vpr_M5_full/g3vpr_universal_best.pth"
if os.path.isfile(m5_ckpt):
    subprocess.run([
        sys.executable, "-u", "eval_cvpr_v3_suite.py",
        "--ckpt", m5_ckpt,
        "--checkpoints_dir", "./checkpoints",
        "--query_bs", "1",
        "--run_sota_hub",
        "--run_faiss_1m",
        "--output_dir", "./results",
    ], check=True)
    subprocess.run([
        sys.executable, "-u", "experiments/visualize_gaussians.py",
        "--ckpt", m5_ckpt,
    ], check=True)

    report_md = "./results/CVPR_V3_SUMMARY_REPORT.md"
    if os.path.isfile(report_md):
        print("\n" + "=" * 80)
        print(open(report_md, encoding="utf-8").read())
        print("=" * 80)

zip_out = "/kaggle/working/g3vpr_paper_artifacts_v3" if os.path.isdir("/kaggle/working") else "./g3vpr_paper_artifacts_v3"
shutil.rmtree("./export_bundle", ignore_errors=True)
os.makedirs("./export_bundle/results", exist_ok=True)
os.makedirs("./export_bundle/checkpoints/g3vpr_M5_full", exist_ok=True)
if os.path.isdir("./results"):
    shutil.copytree("./results", "./export_bundle/results", dirs_exist_ok=True)

for main_fn in ("g3vpr_universal_best.pth", "g3vpr_best_fp16.pth", "training_metrics.json"):
    src_p = f"./checkpoints/g3vpr_M5_full/{main_fn}"
    if os.path.isfile(src_p):
        shutil.copy2(src_p, f"./export_bundle/checkpoints/g3vpr_M5_full/{main_fn}")

for entry in sorted(os.listdir("./checkpoints")):
    if entry.startswith("ablation_"):
        abl_src = os.path.join("./checkpoints", entry)
        abl_dst = os.path.join("./export_bundle/checkpoints", entry)
        os.makedirs(abl_dst, exist_ok=True)
        for fn in ("g3vpr_best_fp16.pth", "training_metrics.json"):
            fp = os.path.join(abl_src, fn)
            if os.path.isfile(fp):
                shutil.copy2(fp, os.path.join(abl_dst, fn))

archive_file = shutil.make_archive(zip_out, "zip", "./export_bundle")
print("[HOAN TAT G3-VPR V3] Tai file ket qua + bao cao doi chieu tai Output:", archive_file)